# Data Cleaning and Formatting


### 1. Importing libraries

In [35]:
import pandas as pd
import numpy as np

### 2. Loading Datasets

In [36]:
accounts=pd.read_csv('../data/ravenstack_accounts.csv')
subscriptions=pd.read_csv('../data/ravenstack_subscriptions.csv')
feature_usage = pd.read_csv('../data/ravenstack_feature_usage.csv')
support_tickets = pd.read_csv('../data/ravenstack_support_tickets.csv')
churn_events = pd.read_csv('../data/ravenstack_churn_events.csv') 

### 3. Defining Functions

In [37]:
# Handle missing values
def handle_missing(df, strategy="drop", fill_value=None):
    if strategy == "drop":
        df = df.dropna()
        print("Dropped rows with missing values.")
    elif strategy == "fill":
        df = df.fillna(fill_value)
        print(f"Filled missing values with {fill_value}.")
    else:
        print("No missing value operation performed.")
    return df


In [38]:
#Remove duplicates
def remove_duplicates(df):
    before = df.shape[0]
    df = df.drop_duplicates()
    after = df.shape[0]
    print(f"Removed {before - after} duplicate rows.")
    return df

In [39]:
#Convert to datetime
def convert_to_datetime(df, cols, dayfirst=False):
    for col in cols:
        df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=dayfirst)
    print(f"Converted columns {cols} to datetime.")
    return df

In [40]:
#Standardize Column names
def standardize_column_names(df):
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )
    return df

In [41]:
#clean_categorical
def clean_categorical(df, cols):
    for col in cols:
        df[col] = df[col].astype(str).str.strip().str.lower()
        df[col] = df[col].replace("nan", "unknown")
    print(f"Cleaned categorical columns: {cols}")
    return df

In [42]:
#Summarize
def cleaning_summary(df):
    print("DataFrame Shape:", df.shape)
    print("\nMissing values per column:\n", df.isnull().sum())
    print("\nData types:\n", df.dtypes)
    print("\nDuplicate rows:", df.duplicated().sum())

### 4. Cleaning each datasets

#### A. Accounts Dataset

In [43]:
accounts = standardize_column_names(accounts)
accounts = remove_duplicates(accounts)
accounts = convert_to_datetime(accounts, ["signup_date"], dayfirst=True)
accounts = clean_categorical(accounts, ["industry", "country", "plan_tier", "referral_source"])

Removed 0 duplicate rows.
Converted columns ['signup_date'] to datetime.
Cleaned categorical columns: ['industry', 'country', 'plan_tier', 'referral_source']


#### B. Subscriptions Dataset

In [44]:
subscriptions = standardize_column_names(subscriptions)
subscriptions = remove_duplicates(subscriptions)
subscriptions = convert_to_datetime(subscriptions, ["start_date", "end_date"], dayfirst=True)
subscriptions = clean_categorical(subscriptions, ["plan_tier", "billing_frequency"])
subscriptions = handle_missing(subscriptions, strategy="fill", fill_value=pd.NaT)


Removed 0 duplicate rows.
Converted columns ['start_date', 'end_date'] to datetime.
Cleaned categorical columns: ['plan_tier', 'billing_frequency']
Filled missing values with NaT.


#### C. Feature_usage Dataset

In [45]:
feature_usage = standardize_column_names(feature_usage)
feature_usage = remove_duplicates(feature_usage)
feature_usage = convert_to_datetime(feature_usage, ["usage_date"], dayfirst=True)
feature_usage = clean_categorical(feature_usage, ["feature_name"])
feature_usage = handle_missing(feature_usage, strategy="none")

# Remove invalid rows
feature_usage = feature_usage[
    (feature_usage["usage_count"] >= 0) &
    (feature_usage["usage_duration_secs"] >= 0) &
    (feature_usage["error_count"] >= 0)
]

Removed 0 duplicate rows.
Converted columns ['usage_date'] to datetime.
Cleaned categorical columns: ['feature_name']
No missing value operation performed.


#### D. Support_tickets Dataset

In [46]:
support_tickets = standardize_column_names(support_tickets)
support_tickets = remove_duplicates(support_tickets)
support_tickets = convert_to_datetime(support_tickets, ["submitted_at", "closed_at"], dayfirst=True)
support_tickets = handle_missing(support_tickets, strategy="fill", fill_value=support_tickets['satisfaction_score'].mean())

# Remove impossible values
support_tickets = support_tickets[support_tickets["resolution_time_hours"] >= 0]


Removed 0 duplicate rows.
Converted columns ['submitted_at', 'closed_at'] to datetime.
Filled missing values with 3.981276595744681.


#### E. Churn_Events Dataset

In [47]:
churn_events = standardize_column_names(churn_events)
churn_events = remove_duplicates(churn_events)
churn_events = convert_to_datetime(churn_events, ["churn_date"], dayfirst=True)
churn_events = clean_categorical(churn_events, ["reason_code"])
churn_events = handle_missing(churn_events, strategy="drop")

# Remove impossible values
churn_events = churn_events[churn_events["refund_amount_usd"] >= 0]

Removed 0 duplicate rows.
Converted columns ['churn_date'] to datetime.
Cleaned categorical columns: ['reason_code']
Dropped rows with missing values.


### 5. Final Sanity Test

In [48]:
datasets = {
    "accounts": accounts,
    "subscriptions": subscriptions,
    "support_tickets": support_tickets,
    "feature_usage": feature_usage,
    "churn_events": churn_events
}

for name, df in datasets.items():
    print(f"\n{name.upper()} dataset:")
    print(df.info())
    print("Missing values:\n", df.isnull().sum())
    print("-"*50)


ACCOUNTS dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   account_id       500 non-null    object        
 1   account_name     500 non-null    object        
 2   industry         500 non-null    object        
 3   country          500 non-null    object        
 4   signup_date      500 non-null    datetime64[ns]
 5   referral_source  500 non-null    object        
 6   plan_tier        500 non-null    object        
 7   seats            500 non-null    int64         
 8   is_trial         500 non-null    bool          
 9   churn_flag       500 non-null    bool          
dtypes: bool(2), datetime64[ns](1), int64(1), object(6)
memory usage: 32.4+ KB
None
Missing values:
 account_id         0
account_name       0
industry           0
country            0
signup_date        0
referral_source    0
plan_tier    